In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


In [ ]:
import matplotlib.pyplot as plt
from configs.config import sys_config
from configs.nn_setting import nn_setting
from tqdm import tqdm
from matplotlib import cm


In [ ]:
import torch
import sys

from data.loaders import load_case, load_measurement, load_load_pv, load_dataset

from models.baselines.sota_gcn import GCN

from data.dataset import rnn_dataset

from torch.utils.data import DataLoader
import numpy as np
from torch import optim
from data.dataset import scaler
from time import time


In [ ]:
class Evaluation:
    def __init__(self, case_class):

        self.nn_setting = nn_setting
        self.case_class = case_class
        self.case_class.gen_torch_constant()

        # Load the model and trained parameters
        model = GCN().to(nn_setting['device'])
        model_path = f"saved_model/{sys_config['case_name']}/SOTA_GCN.pt"

        model.load_state_dict(torch.load(model_path,
            map_location=torch.device(nn_setting['device']), weights_only=True))

        device = nn_setting['device']
        print(f'Using {device} device')
        model.to(device)
        model.eval()
        print(model)

        self.model = model
        self.device = device

        # Scaling factor
        self.scaler_ = scaler()
        self.t_min = torch.tensor(self.scaler_.min, device=device, dtype=torch.float32)
        self.t_max = torch.tensor(self.scaler_.max, device=device, dtype=torch.float32)
        self.t_diff = self.t_max - self.t_min + 1e-7

        # Load Validation dataset for threshold calibration
        valid_set = rnn_dataset(mode='valid', istransform=True)
        valid_loader = DataLoader(valid_set, batch_size=1, shuffle=False)

        # Calculate GCN scores on validation (normal) data
        valid_scores = []
        with torch.no_grad():
            for input, output in valid_loader:
                assert np.all(input.numpy() == output.numpy())
                input = input.to(device)
                logit = model(input)
                prob = torch.sigmoid(logit).cpu().item()
                valid_scores.append(prob)

        # Detection thresholds (higher score = more likely attack)
        # Use the physics-guided notebook's threshold quantile list.
        quantile = [0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.999]
        ae_threshold = np.quantile(valid_scores, quantile)
        self.ae_threshold = ae_threshold
        self.quantile_idx = 6  # Default: quantile 0.92

        print(f'Detection thresholds: {self.ae_threshold}')

        self.quantile = quantile

    def evaluate(self, test_data):
        """
        Evaluate GCN score for a single sample.
        Returns (score,) where score is P(attack) from sigmoid.
        Higher score = more likely attack (matches AE loss semantics).
        """
        if torch.is_tensor(test_data):
            test_data = test_data.float().to(self.device)
        else:
            test_data = torch.tensor(test_data, dtype=torch.float32, device=self.device)

        assert test_data.shape == (1, self.nn_setting['sample_length'],
                                    self.nn_setting['feature_size'])

        self.model.eval()
        with torch.no_grad():
            logit = self.model(test_data)
            score = torch.sigmoid(logit).cpu().item()

        return score


In [ ]:
"""
Test the performance of attack detection and identification algorithms
"""

from configs.config import sys_config, save_metric
from data.loaders import load_case, load_measurement, load_load_pv

# Load cases, measurement, and load
case_class = load_case()
z_noise_summary, v_est_summary = load_measurement()
load_active, load_reactive, pv_active_, pv_reactive_ = load_load_pv()
test_dataloader_scaled, test_dataloader_unscaled, valid_dataloader_scaled, valid_dataloader_unscaled = load_dataset()


dd_detector = Evaluation(case_class=case_class)  # Instance the GCN detector
scaler_ = scaler()                               # Instance the scaler class
print(f'Quantile: {dd_detector.quantile[dd_detector.quantile_idx]}')
print(f'Threshold: {dd_detector.ae_threshold[dd_detector.quantile_idx]}')


In [ ]:
# ---- Compute FP_DDD and TP_DDD for metric_ddd compatibility ----
# FP_DDD: per-sample, True if normal sample is flagged as attack
# TP_DDD: per-sample per-attack-type, True if attack is detected

FP_DDD = []
for input, output in test_dataloader_scaled:
    score = dd_detector.evaluate(input)
    FP_DDD.append(score >= dd_detector.ae_threshold[dd_detector.quantile_idx])

TP_DDD = {}
attack_str = {}
ang_no_list = [1, 2, 3]
ang_str_list = [0.2, 0.3]

for ang_no in ang_no_list:
    for ang_str in ang_str_list:
        key = f'({ang_no},{ang_str})'
        tp_list = []
        att_list = []
        for _, (idx, input, v_est_pre, v_est_last) in tqdm(
            enumerate(test_dataloader_unscaled),
            total=len(test_dataloader_unscaled)
        ):
            v_est_last = v_est_last.flatten()
            z_att_noise, v_att_est_last = case_class.gen_fdi_att_dd(
                z_noise=input, v_est_last=v_est_last,
                ang_no=ang_no, mag_no=0,
                ang_str=ang_str, mag_str=0
            )
            z_att_noise_scale = scaler_(z_att_noise)
            score = dd_detector.evaluate(z_att_noise_scale)
            tp_list.append(score >= dd_detector.ae_threshold[dd_detector.quantile_idx])
            att_list.append(ang_str)
        TP_DDD[key] = tp_list
        attack_str[key] = att_list

metric_ddd = {'FP_DDD': FP_DDD, 'TP_DDD': TP_DDD, 'att_strength': attack_str}


## TPR and FPR

In [ ]:
FP_DD_summary = np.zeros(len(dd_detector.quantile))
for input, output in test_dataloader_scaled:
    score = dd_detector.evaluate(input)
    for idx, quantile in enumerate(dd_detector.quantile):
        if score >= dd_detector.ae_threshold[idx]:
            FP_DD_summary[idx] += 1

FPR_DD_summary = FP_DD_summary / len(test_dataloader_scaled.dataset)


print(f'GCN FPR: {FPR_DD_summary}')


In [ ]:
# Attack list
ang_no_list = [1,2,3]
mag_no = 0
ang_str_list = [0.2,0.3]
mag_str = 0

ROC_dict = {}

for ang_no in ang_no_list:
    # Attack number
    for ang_str in ang_str_list:
        # Attack strength
        
        residual = []
        for _, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):
            # Convert data format
            v_est_last = v_est_last.flatten()
            # Attack
            z_att_noise, v_att_est_last = case_class.gen_fdi_att_dd(z_noise=input, v_est_last=v_est_last, ang_no=ang_no, mag_no=mag_no, ang_str=ang_str, mag_str=mag_str)
            # Scale
            z_att_noise_scale = scaler_(z_att_noise)
            score = dd_detector.evaluate(z_att_noise_scale)
            residual.append(score)
        
        TPR_DDD_summary = []
        # For each threshold, find detection rate
        for i, quantile in enumerate(dd_detector.quantile):
            TPR_DDD = np.sum(np.array(residual) >= dd_detector.ae_threshold[i])/len(residual)
            TPR_DDD_summary.append(TPR_DDD)
            
        ROC_dict[f'({ang_no},{ang_str})'] = TPR_DDD_summary


In [ ]:
plt.figure()
plt.plot(1-np.array(dd_detector.quantile), FPR_DD_summary, label = 'Test Set', color = cm.Dark2(0))
plt.plot(1-np.array(dd_detector.quantile), 1-np.array(dd_detector.quantile), label = 'Valid Set', linestyle = 'dashed', color = cm.Dark2(0))
plt.vlines(x = 1-dd_detector.quantile[dd_detector.quantile_idx], ymin = 0, ymax = 1, colors = 'red', linestyles='--', label='Threshold')
plt.xlabel('FPR')
plt.ylabel('FPR')
plt.fill_between(1-np.array(dd_detector.quantile), 1-np.array(dd_detector.quantile), FPR_DD_summary, color = cm.Dark2(0), alpha = 0.1)
plt.legend()
plt.ylim([0.0,0.31])
plt.yticks(np.arange(0,0.31,0.05))
plt.xlim([0.0,0.2])
plt.grid()
plt.savefig(f'figures/{sys_config["case_name"]}/gcn_false_positive.pdf')
plt.show()


In [ ]:
# ROC curve
plt.figure()
for idx, key in enumerate(ROC_dict.keys()):
    plt.plot(FPR_DD_summary, ROC_dict[key], label = key, color = cm.Dark2(idx), linestyle = '-', marker = 'o')
    if idx == 3:
        break
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.grid()
plt.vlines(x = FPR_DD_summary[dd_detector.quantile_idx], ymin = 0, ymax = 1, colors = 'red', linestyles='--', label='Threshold')
plt.legend()
plt.ylim([0.45,1.0])
plt.xlim([0,0.30])
plt.savefig(f'figures/{sys_config["case_name"]}/gcn_roc.pdf')
plt.show()

print(f'The validation set FPR: {1-dd_detector.quantile[dd_detector.quantile_idx]}')
print(f'The false positive rate is: {np.sum(metric_ddd["FP_DDD"])/len(metric_ddd["FP_DDD"])}')


In [ ]:

TPR_DDD = []
attack_str = []
for key in metric_ddd['TP_DDD'].keys():
    TPR_DDD.append(np.sum(metric_ddd['TP_DDD'][key])/len(metric_ddd['TP_DDD'][key]))
    attack_str.append(metric_ddd['att_strength'][key])
print(TPR_DDD)

plt.plot(TPR_DDD)
plt.title('The TPR of GCN')
plt.grid()
plt.show()

print(f'minimum attack strength: {np.min(attack_str,1)}')


In [ ]:
## AUC, F1-score, ACC, FPR, MCC

from sklearn.metrics import auc, precision_score, recall_score, f1_score, accuracy_score, matthews_corrcoef
import numpy as np

for i in range(0,4):
    try:
        model_key = list(ROC_dict.keys())[i]
        tpr_array = ROC_dict[model_key]
        fpr_array = FPR_DD_summary
        roc_auc = auc(fpr_array, tpr_array)
        print(f'AUC-ROC (Attack Case: {model_key}): {roc_auc:.4f}')
    except NameError:
        print('Warning: ROC_dict or FPR_DD_summary not defined.')
    except IndexError:
        print('Warning: ROC_dict is empty.')


# --- F1-score, ACC, FPR, MCC (based on selected threshold) ---
all_tpr_results = []
for key in metric_ddd['TP_DDD'].keys():
    all_tpr_results.extend(metric_ddd['TP_DDD'][key])

TP_FN_count = len(all_tpr_results)
TP_count = np.sum(all_tpr_results)
FN_count = TP_FN_count - TP_count

FP_TN_count = len(metric_ddd['FP_DDD'])
FP_count = np.sum(metric_ddd['FP_DDD'])
TN_count = FP_TN_count - FP_count

Y_true = np.concatenate([np.ones(TP_FN_count), np.zeros(FP_TN_count)])
all_fp_results = metric_ddd['FP_DDD']
Y_pred = np.concatenate([np.array(all_tpr_results), np.array(all_fp_results)])

accuracy = accuracy_score(Y_true, Y_pred)
precision = precision_score(Y_true, Y_pred, zero_division=0)
recall = recall_score(Y_true, Y_pred, zero_division=0)
f1 = f1_score(Y_true, Y_pred, zero_division=0)
fpr = FP_count / FP_TN_count if FP_TN_count > 0 else 0.0
mcc = matthews_corrcoef(Y_true, Y_pred)

print('-' * 45)
print(f'Metrics for selected Threshold (Quantile 0.92):')
print('-' * 45)
print(f'  [Sample Counts]')
print(f'    Total Attacked Samples (TP+FN): {TP_FN_count}')
print(f'    Total Normal Samples (FP+TN)  : {FP_TN_count}')
print(f'    True Positives (TP)           : {TP_count}')
print(f'    False Positives (FP)          : {FP_count}')
print(f'    True Negatives (TN)           : {TN_count}')
print(f'    False Negatives (FN)          : {FN_count}')
print(f'  [Evaluation Metrics]')
print(f'    Accuracy (ACC)                : {accuracy:.4f}')
print(f'    Precision                     : {precision:.4f}')
print(f'    Recall (TPR)                  : {recall:.4f}')
print(f'    False Positive Rate (FPR)     : {fpr:.4f}')
print(f'    F1-Score                      : {f1:.4f}')
print(f'    Matthews Corr Coeff (MCC)     : {mcc:.4f}')
print('-' * 45)


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(Y_true, Y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print('-' * 30)
print('Confusion Matrix:')
print(f' [TN: {tn}]  [FP: {fp}]')
print(f' [FN: {fn}]  [TP: {tp}]')
print('-' * 30)

plt.figure(figsize=(6, 5))
labels = ['Normal (0)', 'Attack (1)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, annot_kws={'size': 16})

plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title('GCN: Confusion Matrix', fontsize=14)
plt.show()

if tn == 0 and fp > 0:
    print('WARNING: TN = 0, all normal samples flagged as attack!')
if tp == 0 and fn > 0:
    print('WARNING: TP = 0, no attacks detected!')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import auc

num_neg = len(test_dataloader_scaled.dataset)
num_pos = len(test_dataloader_unscaled.dataset)

model_key = list(ROC_dict.keys())[0]
tpr_array = np.array(ROC_dict[model_key])
fpr_array = np.array(FPR_DD_summary)

sort_idx = np.argsort(fpr_array)
fpr_sorted = fpr_array[sort_idx]
tpr_sorted = tpr_array[sort_idx]

roc_auc = auc(fpr_sorted, tpr_sorted)

tp_count = tpr_sorted * num_pos
fp_count = fpr_sorted * num_neg
epsilon = 1e-8

precision_array = tp_count / (tp_count + fp_count + epsilon)
recall_array = tpr_sorted

f1_array = 2 * (precision_array * recall_array) / (precision_array + recall_array + epsilon)

max_f1_idx = np.argmax(f1_array)
max_f1 = f1_array[max_f1_idx]

original_best_idx = sort_idx[max_f1_idx]
best_threshold = dd_detector.ae_threshold[original_best_idx]
best_quantile = dd_detector.quantile[original_best_idx]

print(f'Model Scenario: {model_key}')
print(f'AUC-ROC: {roc_auc:.4f}')
print(f'Max F1-Score: {max_f1:.4f}')
print(f'Best Threshold (from F1): {best_threshold:.4f} (Quantile: {best_quantile})')

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(fpr_sorted, tpr_sorted, label=f'AUC = {roc_auc:.2f}', marker='o')
plt.title('GCN: ROC Curve')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(fpr_sorted, f1_array, label='F1-Score', color='orange', marker='x')
plt.scatter(fpr_sorted[max_f1_idx], max_f1, color='red', s=100, label='Max F1')
plt.title(f'GCN: F1-Score (Max: {max_f1:.2f})')
plt.xlabel('FPR')
plt.ylabel('F1 Score')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()
